# DataRetail 360 · Integración analítica confiable
## Evaluación Parcial 3 — IDY1102
**Estructuras y Visualización de Datos · Oscar Muñoz**  
**Entorno:** PostgreSQL de origen → Python/Pandas → SQL Server 2025.  
**Modalidad:** notebook técnico, ejecutable y documentado, con historial Git local.

Este trabajo integra las fuentes del caso para disponer información consistente de ventas,
clientes, productos, inventario y campañas. La empresa presenta cifras contradictorias porque
mezcla estados, impuestos, devoluciones, horarios y fuentes duplicadas. El objetivo de esta etapa
es construir la base analítica y demostrar calidad, integridad e historia temporal antes del consumo BI.

### Ruta de lectura y evaluación
| Criterio (20% cada uno) | Evidencia prevista |
|---|---|
| Modelo dimensional | Bus Matrix, diagrama, grano y diccionario |
| SCD | Tipos 1/2/3, justificación y pruebas temporales |
| Pipeline | Extracción, contratos, transformación, rechazos y logging |
| Almacenamiento | Bronze/Silver/Gold persistentes y SQL Server |
| Documentación | Decisiones, resultados, pruebas, linaje y Git |

### Ejecución
Crear el entorno con `python -m venv .venv`, instalar `requirements.txt`, seleccionar ese
intérprete en Jupyter/VS Code y completar `.env` a partir de `.env.example`.
Requiere PostgreSQL local poblado y SQL Server local con autenticación Windows y ODBC Driver 18.
Ejecutar las celdas en orden. La base `DataRetail360_DW` fue autorizada y creada previamente.
Los datos y logs se guardan en carpetas excluidas de Git. No se imprimen clientes ni credenciales.
El cuaderno es el entregable principal; el entorno, las bases y los archivos generados son sus dependencias.

## 1. Fuentes, alcance y decisiones
La fuente existente `idy1102_ep2` contiene 16 tablas `dr360_raw` que simulan los CSV,
Excel y sistemas del negocio. Se leen como tablas SQL; no se afirma haber recibido archivos
CSV/Excel independientes. Las cinco tablas `dr360_control` describen la simulación.
Las claves de persona real del generador **no se usan para resolver identidades**: hacerlo
convertiría una respuesta conocida de la simulación en una capacidad inexistente del negocio.

Se preservan las tablas de origen. La extracción se realiza en una transacción de solo lectura,
con una instantánea consistente. Las transformaciones ocurren en Pandas: es ETL hacia el DWH,
con conservación previa de crudos en Bronze. Medallion describe las capas de calidad, no obliga a usar Spark.

### Bus Matrix y granularidad
| Proceso | Una fila representa | Fecha | Producto | Sucursal | Cliente | Canal | Campaña |
|---|---|---|---|---|---|---|---|
| Ventas | Línea de documento + sistema POS/ECOM | Sí | Sí | Sí | Sí | Sí | Sí |
| Devoluciones | Registro de devolución | Sí | Sí | Sí | Sí | Sí | No |
| Inventario | Corte semanal × sucursal × producto | Sí | Sí | Sí | No | No | No |
| Campañas | Envío a un registro CRM | Sí | No | No | Sí | No | Sí |
| Web | Sesión | Sí | Producto principal | No | Sí | No | UTM |
| Reposición | Orden de reposición | Solicitud/recepción | Sí | Sí | No | No | No |

Las tablas de hechos no se unen directamente entre sí: se agregan al mismo grano antes de
comparar. El stock es semiaditivo: se suma entre productos/sucursales, **no entre cortes**.
La tabla de envíos no identifica un producto específico; no se inventa esa relación.
Los reportes legacy tienen grano agregado y quedan en una zona independiente de conciliación.

### Dimensiones y jerarquías
`dim_fecha`: día → mes → trimestre → año. `dim_producto`: producto → subcategoría → categoría
→ línea; marca es un atributo paralelo, no un nivel superior de categoría.
`dim_sucursal` y `dim_cliente`: comuna → región → país con roles geográficos distintos.
No se inventan provincia ni coordenadas que no existen en la fuente.
Se elige una constelación de estrellas con dimensiones conformadas; un copo de nieve separaría
geografías/categorías, pero añadiría joins sin necesidad para este volumen y uso académico.
Los miembros desconocidos usan clave 0; el canal digital sin local usa un miembro explícito.

### Contrato de indicadores
- Venta neta POS: documento EMITIDA, `monto_neto` sin IVA; se distribuye a líneas sin perder pesos.
- Venta neta e-commerce: ENTREGADO; mercadería `(subtotal-descuento)/1,19`, sin despacho,
  redondeada a CLP. La tasa 19% es un **supuesto del dataset**, no asesoría tributaria.
- Devoluciones: APROBADA y trazable al documento/producto; se descuentan una sola vez
  en fecha de devolución. Se explicita la base tributaria asumida de `monto_devuelto`.
- DEVUELTO/DEVOLUCION web no se reconoce como venta ENTREGADO. Sus devoluciones no se
  restan otra vez de ventas excluidas. Se informa esta limitación del modelo de estados de la fuente.
- Ticket promedio: venta antes de devoluciones / documentos distintos reconocidos.
- Cliente activo: persona resuelta con compra reconocida en los últimos seis meses respecto
  del corte del dataset; se excluye el cliente desconocido.
- Marketing: conversión reportada y retorno atribuido son indicadores de la plataforma;
  no se presentan como causalidad ni como ingreso conciliado con ventas.
- Inventario: riesgo de reposición según stock de sistema y mínimo; las diferencias con
  stock físico y antigüedad del conteo se reportan, no se corrigen inventando existencias.

Los errores de identidad se resuelven conservadoramente con RUT válido y correo único.
Los emparejamientos ambiguos quedan separados; similitud de nombres no basta para fusionar personas.

### Diagrama del modelo
La siguiente representación resume las relaciones. El archivo `modelo.drawio` contiene el
mismo diseño editable en diagrams.net. Cada hecho conserva su clave natural y las claves
sustitutas de las dimensiones aplicables.

```text
Fecha ─────┬── fact_ventas ───── Producto (SCD2)
Canal ─────┤       │                │
Campaña ───┤       ├── Sucursal ────┼── fact_inventario
Cliente ───┘       │                └── fact_reposicion
 (SCD2)            └── fact_devoluciones
   │
   ├── fact_marketing ── Campaña
   └── fact_web ──────── Producto / Campaña
```

SCD1 corrige atributos descriptivos; SCD2 conserva versiones de región/segmento del cliente
y categoría del producto; SCD3 mantiene zona actual/anterior de la sucursal.
La fuente solo contiene el estado actual: la primera carga **no reconstruye historia real**.
Se marca esa limitación y se demuestran cambios con casos controlados separados del negocio.

## 2. Extracción y capa Bronze

Se preserva una instantánea consistente del origen. Los hashes identifican lotes y evitan sobrescribir crudos; el manifiesto permite reconstruir el linaje. Un error de esquema detiene el proceso. Solo se reintentan errores de conexión.

In [ ]:
# Contrato explícito obtenido del DDL proporcionado.
CONTRACT = {'pos_sucursales': ['id_sucursal', 'codigo_sucursal', 'nombre_sucursal', 'formato', 'direccion', 'comuna', 'region', 'zona', 'superficie_m2', 'fecha_apertura', 'fecha_cierre', 'estado', '_origen', '_fecha_carga'], 'pos_productos': ['codigo_producto', 'descripcion', 'linea_negocio', 'categoria', 'subcategoria', 'marca', 'codigo_barra', 'precio_lista', 'costo_unitario', 'estado', 'fecha_alta', 'fecha_baja', '_origen', '_fecha_carga'], 'pos_ventas_cabecera': ['id_boleta', 'tipo_documento', 'nro_documento', 'codigo_sucursal', 'nro_caja', 'fecha_hora', 'rut_cliente', 'medio_pago', 'codigo_cupon', 'monto_bruto', 'descuento_total', 'monto_neto', 'monto_iva', 'monto_total', 'estado', 'id_vendedor', '_origen', '_fecha_carga'], 'pos_ventas_detalle': ['id_boleta', 'nro_linea', 'codigo_producto', 'cantidad', 'precio_unitario', 'descuento', 'total_linea', '_origen', '_fecha_carga'], 'ecom_catalogo': ['sku_web', 'nombre_web', 'categoria_web', 'marca', 'vendedor', 'precio_web', 'codigo_pos_ref', 'ean', 'activo_web', 'fecha_publicacion', '_origen', '_fecha_carga'], 'ecom_cuentas': ['id_cuenta', 'email', 'nombre_completo', 'rut', 'telefono', 'comuna', 'fecha_creacion', 'ultimo_login', 'acepta_newsletter', '_origen', '_fecha_carga'], 'ecom_pedidos': ['id_pedido', 'id_cuenta', 'email_contacto', 'id_sesion', 'canal_venta', 'fecha_pedido_utc', 'estado_pedido', 'medio_pago', 'tipo_entrega', 'codigo_sucursal_retiro', 'comuna_despacho', 'codigo_cupon', 'subtotal', 'descuento', 'costo_despacho', 'total_pagado', 'fecha_entrega', '_origen', '_fecha_carga'], 'ecom_pedidos_detalle': ['id_pedido', 'nro_linea', 'sku_web', 'cantidad', 'precio_unitario', 'descuento', 'total_linea', '_origen', '_fecha_carga'], 'web_sesiones': ['id_sesion', 'id_cuenta', 'fecha_inicio_utc', 'dispositivo', 'fuente_trafico', 'utm_campaign', 'paginas_vistas', 'duracion_seg', 'sku_principal_visto', 'agrego_carrito', 'inicio_checkout', 'compro', '_origen', '_fecha_carga'], 'crm_clientes': ['id_cliente', 'rut', 'nombres', 'apellido_paterno', 'apellido_materno', 'email', 'telefono', 'fecha_nacimiento', 'genero', 'comuna', 'region', 'segmento_crm', 'canal_registro', 'fecha_registro', 'acepta_marketing', 'estado', '_origen', '_fecha_carga'], 'mkt_campanias': ['id_campania', 'codigo_campania', 'nombre_campania', 'tipo_campania', 'canal_objetivo', 'categoria_foco', 'segmento_objetivo', 'descuento_pct', 'fecha_inicio', 'fecha_fin', 'presupuesto_clp', 'responsable', '_origen', '_fecha_carga'], 'mkt_envios_campania': ['id_envio', 'id_campania', 'id_cliente', 'canal_envio', 'fecha_envio', 'entregado', 'abierto', 'fecha_apertura', 'click', 'convertido', 'monto_atribuido', '_origen', '_fecha_carga'], 'inv_stock_semanal': ['fecha_corte', 'codigo_sucursal', 'codigo_producto', 'stock_sistema', 'stock_fisico', 'stock_minimo', 'punto_reposicion', 'unidades_en_transito', 'fecha_ultimo_conteo', '_origen', '_fecha_carga'], 'inv_ordenes_reposicion': ['id_orden', 'codigo_sucursal', 'codigo_producto', 'fecha_solicitud', 'fecha_recepcion', 'cantidad_solicitada', 'cantidad_recibida', 'estado', '_origen', '_fecha_carga'], 'devoluciones': ['id_devolucion', 'canal_origen', 'id_documento_origen', 'codigo_producto', 'fecha_devolucion', 'cantidad', 'monto_devuelto', 'motivo', 'codigo_sucursal_recepcion', 'estado', '_origen', '_fecha_carga'], 'legacy_reporte_ventas_xlsx': ['archivo_origen', 'hoja', 'fila_excel', 'fecha', 'sucursal', 'linea', 'unidades', 'monto_venta', 'estado', 'responsable', 'observacion', '_origen', '_fecha_carga']}

In [ ]:
# Configuración: credenciales fuera del notebook, rutas relativas y versiones verificables.
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, logging, uuid, time, re, unicodedata, subprocess
from decimal import Decimal, ROUND_HALF_UP
from contextlib import contextmanager
import pandas as pd
import numpy as np
import sqlalchemy as sa
from sqlalchemy import text
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'EP3_DataRetail360.ipynb').exists():
    raise RuntimeError('Ejecutar desde la carpeta DataRetail360_EP3.')
load_dotenv(ROOT / '.env', encoding='utf-8-sig')
if not os.getenv('PGPASSWORD'):
    from getpass import getpass
    os.environ['PGPASSWORD'] = getpass('Contraseña PostgreSQL (no se muestra): ')
DATA = ROOT / 'data'
for folder in ['bronze', 'silver', 'gold', 'control', 'rechazos']:
    (DATA / folder).mkdir(parents=True, exist_ok=True)
(ROOT / 'logs').mkdir(exist_ok=True)
RUN_ID = uuid.uuid4().hex
LOG = logging.getLogger('dr360')
LOG.setLevel(logging.INFO)
for handler in LOG.handlers[:]:
    handler.close(); LOG.removeHandler(handler)
handler = logging.FileHandler(ROOT / 'logs' / f'{RUN_ID}.log', encoding='utf-8')
handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
LOG.addHandler(handler)
LOG.info('Inicio run=%s', RUN_ID)

PG = sa.create_engine(sa.URL.create('postgresql+psycopg',
    username=os.getenv('PGUSER','postgres'), password=os.environ['PGPASSWORD'],
    host=os.getenv('PGHOST','localhost'), port=int(os.getenv('PGPORT','5432')),
    database=os.getenv('PGDATABASE','idy1102_ep2')), hide_parameters=True,
    connect_args={'connect_timeout':10}, pool_pre_ping=True)
DW = sa.create_engine(sa.URL.create('mssql+pyodbc',
    host=os.getenv('MSSQL_SERVER','localhost'), database=os.getenv('MSSQL_DATABASE','DataRetail360_DW'),
    query={'driver':'ODBC Driver 18 for SQL Server','Trusted_Connection':'yes',
           'Encrypt':'yes','TrustServerCertificate':'yes'}),
    fast_executemany=True, hide_parameters=True, pool_pre_ping=True)
# TrustServerCertificate es una decisión del laboratorio local, no de producción.
with DW.connect() as conn:
    assert conn.scalar(text('SELECT DB_NAME()')) == 'DataRetail360_DW'
print('Conexiones configuradas; SQL Server disponible. Run:', RUN_ID)

# Natural keys de fuentes: permiten un orden reproducible y controles de duplicidad.
KEYS = {
 'pos_sucursales':['codigo_sucursal'], 'pos_productos':['codigo_producto'],
 'pos_ventas_cabecera':['id_boleta'], 'pos_ventas_detalle':['id_boleta','nro_linea'],
 'ecom_catalogo':['sku_web'], 'ecom_cuentas':['id_cuenta'],
 'ecom_pedidos':['id_pedido'], 'ecom_pedidos_detalle':['id_pedido','nro_linea'],
 'web_sesiones':['id_sesion'], 'crm_clientes':['id_cliente'],
 'mkt_campanias':['id_campania'], 'mkt_envios_campania':['id_envio'],
 'inv_stock_semanal':['fecha_corte','codigo_sucursal','codigo_producto'],
 'inv_ordenes_reposicion':['id_orden'], 'devoluciones':['id_devolucion'],
 'legacy_reporte_ventas_xlsx':['archivo_origen','hoja','fila_excel']}

def extract_snapshot():
    """Reintenta la extracción completa, nunca continúa una instantánea incompleta."""
    for attempt in range(3):
        try:
            raw, inventory = {}, []
            with PG.connect().execution_options(isolation_level='REPEATABLE READ') as conn:
                with conn.begin():
                    conn.execute(text('SET TRANSACTION READ ONLY'))
                    schema = pd.read_sql(text("SELECT table_name,column_name,data_type FROM information_schema.columns WHERE table_schema='dr360_raw' ORDER BY table_name,ordinal_position"), conn)
                    for name, key in KEYS.items():
                        cols = schema.loc[schema.table_name.eq(name),'column_name'].tolist()
                        if not set(CONTRACT[name]).issubset(cols):
                            raise ValueError(f'Contrato roto: {name}')
                        # Identificadores proceden de una lista fija, nunca de entrada del usuario.
                        frame = pd.read_sql(text(f'SELECT * FROM dr360_raw.{name} ORDER BY '+','.join(key)), conn)
                        digest = hashlib.sha256(pd.util.hash_pandas_object(frame,index=False).values.tobytes()).hexdigest()
                        path = DATA/'bronze'/name/f'{digest}.parquet'
                        path.parent.mkdir(exist_ok=True)
                        if not path.exists():
                            temp = path.with_suffix('.tmp'); frame.to_parquet(temp,index=False); temp.replace(path)
                        frame['_src_row'] = np.arange(len(frame),dtype='int64')
                        frame['_batch'] = digest
                        raw[name]=frame
                        inventory.append({'tabla':name,'filas':len(frame),'columnas':len(cols),'hash':digest,'archivo':str(path.relative_to(ROOT))})
                        LOG.info('extract tabla=%s filas=%d',name,len(frame))
                    # Solo metadatos del generador; no se extraen claves de personas reales.
                    controls={n:pd.read_sql(text(f'SELECT * FROM dr360_control.{n}'),conn) for n in
                              ['config_generacion','definiciones_kpi_por_area','problemas_calidad_inyectados']}
            return raw, pd.DataFrame(inventory), schema, controls
        except sa.exc.OperationalError:
            LOG.warning('Falla transitoria de extracción intento=%d',attempt+1)
            if attempt==2: raise
            time.sleep(2**attempt)
        except Exception as exc:
            LOG.error('Extracción detenida tipo=%s',type(exc).__name__)
            raise

raw, inventory, source_schema, controls = extract_snapshot()
manifest={'run_id':RUN_ID,'extracted_at_utc':datetime.now(timezone.utc).isoformat(),
          'sources':inventory.to_dict('records')}
(DATA/'control'/f'{RUN_ID}.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
source_schema.to_parquet(DATA/'control'/'source_schema.parquet',index=False)
display(inventory[['tabla','filas','columnas']])
print('Filas extraídas:',int(inventory.filas.sum()))